<a href="https://colab.research.google.com/github/akulvibhore/flyrank-ml-internship-/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akulvibhore/flyrank-ml-internship-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule

I will prioritize content that has a clear review opportunity: it is old since its last
update and still receives meaningful search visibility. Higher search visibility increases
the priority because improving a visible page can affect more traffic.

The baseline does not use `trend_pct`, `trend_direction`, or `is_declining_label` to create
the score because these are derived from the outcome window.

### Reason codes

- `stale_visible` — content has not been updated for at least 180 days and has at least 300 impressions in 90 days.
- `stale_high_volume` — content has not been updated for at least 180 days and has at least 3,000 impressions in 90 days.
- `visible_low_ctr` — content has at least 300 impressions and its CTR is below the median CTR for its position tier.
- `measurable_opportunity` — content has at least 100 impressions and at least one session.
- `review` — the item is selected by the baseline but does not match one of the more specific reason codes.

In [3]:
# Clone the FlyRank internship repository into Colab

!rm -rf /content/flyrank-ml-internship
!git clone https://github.com/akulvibhore/flyrank-ml-internship-.git /content/flyrank-ml-internship

Cloning into '/content/flyrank-ml-internship'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 132 (delta 42), reused 96 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.88 MiB | 15.49 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [4]:
from pathlib import Path

repo = Path("/content/flyrank-ml-internship")

print("Repository found:", repo.exists())
print("\nCSV files in repository:\n")

csv_files = list(repo.rglob("*.csv"))

for f in csv_files:
    print(f)

Repository found: True

CSV files in repository:

/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [5]:
# ============================================================
# SECTION 1 — LOAD DATA
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Repository location
REPO = Path("/content/flyrank-ml-internship")

# Actual FlyRank dataset
DATA_PATH = REPO / "data" / "raw" / "content_refresh_anonymized.csv"

# Check that the file exists
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Path:", DATA_PATH)
print("Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

Dataset loaded successfully!
Path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:

# ============================================================
# CHECK THE DATASET
# ============================================================

print("Columns and data types:\n")
df.info()

print("\nMissing values:\n")
display(df.isna().sum().sort_values(ascending=False).head(20))

print("\nNumeric summary:\n")
display(df.describe().T)

Columns and data types:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d

,0
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468



Numeric summary:



,count,mean,std,min,25%,50%,75%,max
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.00
competition,27532.0,0.146954,0.285241,0.0,0.0,0.00,0.13,1.00
cpc,27532.0,0.485342,2.101560,0.0,0.0,0.00,0.00,100.36
word_count,22301.0,3107.760325,1452.382598,8.0,2413.0,2877.00,3666.00,9546.00
char_count,22301.0,20665.277835,10115.344042,40.0,15644.0,19116.00,24011.00,111158.00
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.00
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.00,7.00,4178.00
pageviews_90d,30000.0,49.942467,152.101430,0.0,2.0,8.00,33.00,5998.00
sessions_90d,30000.0,37.066633,107.069131,1.0,2.0,7.00,27.00,4345.00
users_90d,30000.0,35.937700,103.748185,1.0,2.0,7.00,27.00,4913.00


In [7]:
# ============================================================
# CHECK REQUIRED ML-07 COLUMNS
# ============================================================

possible_columns = [
    "content_id",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "trend_pct",
    "trend_direction",
    "is_declining_label",
]

print("ML-07 column check:\n")

for col in possible_columns:
    print(f"{col:25} ->", "FOUND" if col in df.columns else "NOT FOUND")

ML-07 column check:

content_id                -> FOUND
impressions_90d           -> FOUND
sessions_90d              -> FOUND
ctr                       -> FOUND
avg_position              -> FOUND
days_since_last_update    -> FOUND
trend_pct                 -> FOUND
trend_direction           -> FOUND
is_declining_label        -> NOT FOUND


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO = Path("/content/flyrank-ml-internship")

DATA_PATH = REPO / "data" / "raw" / "content_refresh_anonymized.csv"

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Build the ranked queue (writes the CSV)

### Baseline scoring rule

I score each item using simple observable signals.

A stale item gets 2 points when it has not been updated for at least 180 days.
An item gets 1 point for moderate visibility, 1 additional point for high visibility,
1 point when it has measurable opportunity, and 1 point when CTR is below the
median CTR for its position tier.

The final score is used only to prioritize manual review. It does not prove that
an action will improve performance.

In [9]:
# ============================================================
# SECTION 2 — BUILD THE RANKED QUEUE
# ============================================================

work = df.copy()

# ------------------------------------------------------------
# 1. Clean numeric columns
# ------------------------------------------------------------

numeric_cols = [
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

for col in numeric_cols:
    work[col] = pd.to_numeric(
        work[col],
        errors="coerce"
    ).fillna(0)


# ------------------------------------------------------------
# 2. Create simple signals
# ------------------------------------------------------------

# Old / stale content
work["stale_flag"] = (
    work["days_since_last_update"] >= 180
).astype(int)

# Moderate visibility
work["visible_flag"] = (
    work["impressions_90d"] >= 300
).astype(int)

# High visibility
work["high_volume_flag"] = (
    work["impressions_90d"] >= 3000
).astype(int)

# Measurable opportunity
work["opportunity_flag"] = (
    (work["impressions_90d"] >= 100) &
    (work["sessions_90d"] > 0)
).astype(int)


# ------------------------------------------------------------
# 3. CTR vs position
# ------------------------------------------------------------

# Create simple position tiers
def position_tier(position):
    if position <= 0:
        return "unknown"
    elif position <= 3:
        return "top_3"
    elif position <= 10:
        return "4_10"
    elif position <= 20:
        return "11_20"
    else:
        return "21_plus"

work["position_tier"] = work["avg_position"].apply(position_tier)


# Median CTR within each position tier
valid_position = work["position_tier"] != "unknown"

tier_median_ctr = (
    work.loc[valid_position]
    .groupby("position_tier")["ctr"]
    .median()
)

work["tier_median_ctr"] = (
    work["position_tier"]
    .map(tier_median_ctr)
    .fillna(0)
)


# Low CTR relative to position
work["low_ctr_flag"] = (
    (work["position_tier"] != "unknown") &
    (work["impressions_90d"] >= 300) &
    (work["ctr"] < work["tier_median_ctr"])
).astype(int)


# ------------------------------------------------------------
# 4. Calculate transparent action score
# ------------------------------------------------------------

work["action_score"] = (
    2 * work["stale_flag"]
    + work["visible_flag"]
    + work["high_volume_flag"]
    + work["opportunity_flag"]
    + work["low_ctr_flag"]
)


# ------------------------------------------------------------
# 5. Create reason codes
# ------------------------------------------------------------

def make_reason(row):

    reasons = []

    if row["stale_flag"] and row["high_volume_flag"]:
        reasons.append("stale_high_volume")

    elif row["stale_flag"] and row["visible_flag"]:
        reasons.append("stale_visible")

    if row["low_ctr_flag"]:
        reasons.append("visible_low_ctr")

    if row["opportunity_flag"]:
        reasons.append("measurable_opportunity")

    if not reasons and row["action_score"] > 0:
        reasons.append("review")

    if not reasons:
        return "no_signal"

    return "|".join(reasons)


work["reason_code"] = work.apply(
    make_reason,
    axis=1
)


# ------------------------------------------------------------
# 6. Rank everything
# ------------------------------------------------------------

work = work.sort_values(
    by=[
        "action_score",
        "impressions_90d",
        "days_since_last_update"
    ],
    ascending=[
        False,
        False,
        False
    ]
).reset_index(drop=True)

work["rank"] = np.arange(
    1,
    len(work) + 1
)


# ------------------------------------------------------------
# 7. Create output
# ------------------------------------------------------------

OUTPUT_DIR = REPO / "work" / "outputs"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    OUTPUT_DIR /
    "baseline_action_score.csv"
)


output_columns = [
    "rank",
    "content_id",
    "action_score",
    "reason_code",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

queue = work[output_columns].copy()

queue.to_csv(
    OUTPUT_PATH,
    index=False
)


# ------------------------------------------------------------
# 8. Checks
# ------------------------------------------------------------

print("Baseline queue created successfully!")
print("Output:", OUTPUT_PATH)
print("Rows:", len(queue))

print("\nScore distribution:")
display(
    queue["action_score"]
    .value_counts()
    .sort_index()
)

print("\nTop 20:")
display(queue.head(20))

Baseline queue created successfully!
Output: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Rows: 30000

Score distribution:


,count
action_score,
0,7855
1,3241
2,7918
3,9391
4,1585
5,10



Top 20:


,rank,content_id,action_score,reason_code,impressions_90d,sessions_90d,ctr,avg_position,days_since_last_update
0,1,content_cf56e2e2e282,5,stale_high_volume|measurable_opportunity,61678,119,0.15,19.7,194
1,2,content_7368877ea310,5,stale_high_volume|measurable_opportunity,59472,82,0.13,24.8,194
2,3,content_1bfaa38ff26c,5,stale_high_volume|measurable_opportunity,25715,80,0.23,22.2,194
3,4,content_0a91db491d14,5,stale_high_volume|measurable_opportunity,13299,78,0.49,10.5,193
4,5,content_5feee3994adb,5,stale_high_volume|measurable_opportunity,7812,5,0.01,39.0,194
5,6,content_c2d929d83eaa,5,stale_high_volume|measurable_opportunity,7558,25,0.20,17.9,193
6,7,content_b16bd7307b39,5,stale_high_volume|measurable_opportunity,4590,4,0.00,31.0,194
7,8,content_fe16a55cd13d,5,stale_high_volume|measurable_opportunity,4556,42,0.33,16.4,194
8,9,content_ecb6215e79fd,5,stale_high_volume|measurable_opportunity,4429,12,0.38,25.3,194
9,10,content_fd16e3475c29,5,stale_visible|visible_low_ctr|measurable_oppor...,429,1,0.00,9.0,183


In [10]:
# Check that the required CSV was written

output_file = (
    REPO /
    "work" /
    "outputs" /
    "baseline_action_score.csv"
)

print("Exists:", output_file.exists())
print("Path:", output_file)

check = pd.read_csv(output_file)

print("CSV shape:", check.shape)

display(check.head(20))

Exists: True
Path: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
CSV shape: (30000, 9)


,rank,content_id,action_score,reason_code,impressions_90d,sessions_90d,ctr,avg_position,days_since_last_update
0,1,content_cf56e2e2e282,5,stale_high_volume|measurable_opportunity,61678,119,0.15,19.7,194
1,2,content_7368877ea310,5,stale_high_volume|measurable_opportunity,59472,82,0.13,24.8,194
2,3,content_1bfaa38ff26c,5,stale_high_volume|measurable_opportunity,25715,80,0.23,22.2,194
3,4,content_0a91db491d14,5,stale_high_volume|measurable_opportunity,13299,78,0.49,10.5,193
4,5,content_5feee3994adb,5,stale_high_volume|measurable_opportunity,7812,5,0.01,39.0,194
5,6,content_c2d929d83eaa,5,stale_high_volume|measurable_opportunity,7558,25,0.20,17.9,193
6,7,content_b16bd7307b39,5,stale_high_volume|measurable_opportunity,4590,4,0.00,31.0,194
7,8,content_fe16a55cd13d,5,stale_high_volume|measurable_opportunity,4556,42,0.33,16.4,194
8,9,content_ecb6215e79fd,5,stale_high_volume|measurable_opportunity,4429,12,0.38,25.3,194
9,10,content_fd16e3475c29,5,stale_visible|visible_low_ctr|measurable_oppor...,429,1,0.00,9.0,183


## 3. Top-20 review


The top 20 items are a manual review queue. The action is based on the observed
signals behind the score.

Confidence describes how much evidence the rule has, not the probability that
the recommended action will succeed.

Each item also includes what could make the recommendation wrong.

In [11]:
# ============================================================
# SECTION 3 — TOP-20 REVIEW
# ============================================================

top20 = queue.head(20).copy()


def get_action(reason):

    if "stale_high_volume" in reason:
        return "Review and consider updating content"

    if "stale_visible" in reason:
        return "Review freshness and consider updating"

    if "visible_low_ctr" in reason:
        return "Review title/snippet and search-result alignment"

    if "measurable_opportunity" in reason:
        return "Review content for an improvement opportunity"

    return "Manual review"


def get_confidence(score):

    if score >= 5:
        return "Higher confidence: multiple baseline signals"

    elif score >= 3:
        return "Medium confidence: useful signal, needs review"

    else:
        return "Lower confidence: limited signal"


def get_wrong_reason(row):

    if "stale" in row["reason_code"]:
        return (
            "The content may still be accurate despite its age, "
            "so an update may not improve performance."
        )

    if "visible_low_ctr" in row["reason_code"]:
        return (
            "Low CTR may be explained by search position or query mix, "
            "so changing the page may not help."
        )

    if row["impressions_90d"] < 300:
        return (
            "The item has limited visibility, so the opportunity may "
            "be too small to justify action."
        )

    return (
        "The dataset may not contain other factors that affect the "
        "actual action decision."
    )


top20["action"] = top20["reason_code"].apply(
    get_action
)

top20["confidence_note"] = top20["action_score"].apply(
    get_confidence
)

top20["what_would_make_it_wrong"] = top20.apply(
    get_wrong_reason,
    axis=1
)


review_columns = [
    "rank",
    "content_id",
    "action_score",
    "reason_code",
    "action",
    "confidence_note",
    "what_would_make_it_wrong",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

top20_review = top20[review_columns]


display(top20_review)


# Save top-20 review
TOP20_PATH = (
    OUTPUT_DIR /
    "baseline_top20_review.csv"
)

top20_review.to_csv(
    TOP20_PATH,
    index=False
)

print("\nSaved:")
print(TOP20_PATH)

,rank,content_id,action_score,reason_code,action,confidence_note,what_would_make_it_wrong,impressions_90d,sessions_90d,ctr,avg_position,days_since_last_update
0,1,content_cf56e2e2e282,5,stale_high_volume|measurable_opportunity,Review and consider updating content,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,61678,119,0.15,19.7,194
1,2,content_7368877ea310,5,stale_high_volume|measurable_opportunity,Review and consider updating content,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,59472,82,0.13,24.8,194
2,3,content_1bfaa38ff26c,5,stale_high_volume|measurable_opportunity,Review and consider updating content,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,25715,80,0.23,22.2,194
3,4,content_0a91db491d14,5,stale_high_volume|measurable_opportunity,Review and consider updating content,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,13299,78,0.49,10.5,193
4,5,content_5feee3994adb,5,stale_high_volume|measurable_opportunity,Review and consider updating content,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,7812,5,0.01,39.0,194
5,6,content_c2d929d83eaa,5,stale_high_volume|measurable_opportunity,Review and consider updating content,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,7558,25,0.20,17.9,193
6,7,content_b16bd7307b39,5,stale_high_volume|measurable_opportunity,Review and consider updating content,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,4590,4,0.00,31.0,194
7,8,content_fe16a55cd13d,5,stale_high_volume|measurable_opportunity,Review and consider updating content,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,4556,42,0.33,16.4,194
8,9,content_ecb6215e79fd,5,stale_high_volume|measurable_opportunity,Review and consider updating content,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,4429,12,0.38,25.3,194
9,10,content_fd16e3475c29,5,stale_visible|visible_low_ctr|measurable_oppor...,Review freshness and consider updating,Higher confidence: multiple baseline signals,The content may still be accurate despite its ...,429,1,0.00,9.0,183



Saved:
/content/flyrank-ml-internship/work/outputs/baseline_top20_review.csv


## 4. Weak picks + leakage check


The baseline is a prioritization rule, so some top-20 picks can still be weak.

I flag items with limited impressions, a low score, or missing position information
for additional manual review.

The score does not use `trend_pct` or `trend_direction`. These fields describe the
outcome/trend and would create leakage if they were used to decide the ranking.

In [12]:
# ============================================================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# ============================================================

# ------------------------------------------------------------
# 1. Find weak top-20 picks
# ------------------------------------------------------------

weak_mask = (
    (top20["action_score"] <= 2)
    |
    (top20["impressions_90d"] < 300)
    |
    (top20["avg_position"] <= 0)
)

weak_picks = top20.loc[
    weak_mask,
    [
        "rank",
        "content_id",
        "action_score",
        "reason_code",
        "impressions_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]
].copy()

print("Potentially weak Top-20 picks:")

if len(weak_picks) > 0:
    display(weak_picks)
else:
    print("No weak picks found using these checks.")


# ------------------------------------------------------------
# 2. Leakage check
# ------------------------------------------------------------

FORBIDDEN_FEATURES = {
    "trend_pct",
    "trend_direction",
    "is_declining_label"
}

FEATURES_USED_BY_SCORE = {
    "days_since_last_update",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position"
}

leakage = (
    FEATURES_USED_BY_SCORE
    .intersection(FORBIDDEN_FEATURES)
)

print("\nLeakage check")
print("----------------")

print(
    "Forbidden fields present in dataset:",
    sorted(
        FORBIDDEN_FEATURES.intersection(
            set(df.columns)
        )
    )
)

print(
    "Forbidden fields used in score:",
    sorted(leakage)
)

assert len(leakage) == 0, (
    f"Leakage detected: {leakage}"
)

print("\nPASS: No trend/label field is used to calculate action_score.")


# ------------------------------------------------------------
# 3. Confirm score columns
# ------------------------------------------------------------

print("\nActual score inputs:")

for col in sorted(FEATURES_USED_BY_SCORE):
    print(" -", col)


# ------------------------------------------------------------
# 4. Final output check
# ------------------------------------------------------------

assert OUTPUT_PATH.exists()

final_queue = pd.read_csv(
    OUTPUT_PATH
)

assert len(final_queue) == len(df)

assert final_queue["rank"].is_monotonic_increasing

print("\nFINAL CHECK PASSED")
print("------------------")
print("Rows ranked:", len(final_queue))
print("Top score:", final_queue["action_score"].max())
print("Output:", OUTPUT_PATH)


Potentially weak Top-20 picks:
No weak picks found using these checks.

Leakage check
----------------
Forbidden fields present in dataset: ['trend_direction', 'trend_pct']
Forbidden fields used in score: []

PASS: No trend/label field is used to calculate action_score.

Actual score inputs:
 - avg_position
 - ctr
 - days_since_last_update
 - impressions_90d
 - sessions_90d

FINAL CHECK PASSED
------------------
Rows ranked: 30000
Top score: 5
Output: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.